# Exploratory analysis of a news corpus

Walkthrough of the `corpus_tool` pipeline on the *Lee* news corpus (300 short Australian news articles, English) bundled with gensim. Steps: corpus statistics, preprocessing, word frequencies, sentiment, named entities and LDA topic modelling with the number of topics chosen by c_v coherence.

Run after `pip install -r requirements.txt` and `python -m spacy download en_core_web_sm`.

In [1]:
import os, subprocess, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # work from the repository root
sys.path.insert(0, os.getcwd())

import matplotlib.pyplot as plt
import pandas as pd

from corpus_tool import analyzer, extractor, preprocessor

if not os.path.isdir("data/lee_demo"):
    subprocess.run([sys.executable, "scripts/prepare_lee_corpus.py"], check=True)

docs = extractor.load_local_corpus("data/lee_demo")
print(len(docs), "documents")

300 documents


## 1. Raw corpus size

In [2]:
words_per_doc = pd.Series([len(d["text"].split()) for d in docs])
print(words_per_doc.describe().round(1))
words_per_doc.plot.hist(bins=30, title="Words per document (raw)", figsize=(7, 3))
plt.xlabel("words"); plt.show()

count    300.0
mean     199.6
std      104.1
min       45.0
25%      128.8
50%      162.5
75%      252.2
max      620.0
dtype: float64


## 2. Preprocessing

Lemmatization with spaCy, keeping nouns, proper nouns, verbs, adjectives and adverbs; stopwords and very short tokens removed. A few generic news verbs are added as custom stopwords.

In [3]:
nlp = preprocessor.get_nlp("en_core_web_sm")
extra_stop = {"say", "tell", "year", "new", "take", "make", "get"}
tokens = [preprocessor.tokenize(d["text"], nlp, extra_stop) for d in docs]
tokens_kept = [t for t in tokens if len(t) >= 30]
stats = analyzer.lexical_stats(tokens_kept)
pd.Series(stats)

documents                  300.0000
tokens                   29274.0000
vocabulary                5265.0000
type_token_ratio             0.1799
tokens_per_doc_min          30.0000
tokens_per_doc_median       81.0000
tokens_per_doc_max         338.0000
dtype: float64

## 3. Most frequent words

In [4]:
top = analyzer.top_words(tokens_kept, 20)
df = pd.DataFrame(top, columns=["word", "count"])
df.set_index("word")["count"][::-1].plot.barh(figsize=(6, 5), title="Top 20 lemmas")
plt.show()

In [5]:
from wordcloud import WordCloud
cloud = WordCloud(width=900, height=450, background_color="white", colormap="viridis", collocations=False
                  ).generate_from_frequencies(dict(analyzer.top_words(tokens_kept, 200)))
plt.figure(figsize=(9, 4.5)); plt.imshow(cloud); plt.axis("off"); plt.show()

## 4. Sentiment (TextBlob)

TextBlob is lexicon-based and was not validated on this corpus: read it as a rough baseline. News text is mostly neutral, so values cluster near zero.

In [6]:
sent = pd.DataFrame([analyzer.sentiment(d["text"]) for d in docs], columns=["polarity", "subjectivity"])
print(sent.describe().round(3))
sent["polarity"].plot.hist(bins=30, title="Polarity per article", figsize=(7, 3)); plt.show()

       polarity  subjectivity
count   300.000       300.000
mean      0.061         0.373
std       0.095         0.112
min      -0.174         0.000
25%       0.000         0.301
50%       0.062         0.371
75%       0.118         0.441
max       0.441         0.693


## 5. Named entities (spaCy)

In [7]:
ents = analyzer.entities(" ".join(d["text"] for d in docs[:100]), nlp, top=40)
pd.DataFrame([(t, label, n) for (t, label), n in ents], columns=["entity", "type", "count"]).head(20)

,entity,type,count
0,Pakistan,GPE,38
1,two,CARDINAL,33
2,Sydney,GPE,32
3,Afghanistan,GPE,32
4,Palestinian,NORP,32
5,Australia,GPE,27
6,today,DATE,23
7,one,CARDINAL,23
8,US,GPE,22
9,Israeli,NORP,19


## 6. Topic modelling

LDA is fitted for k = 2…10 and the c_v coherence is compared. Coherence is a heuristic: differences between close values of k are small and should not be over-interpreted.

In [8]:
k_best, model, scores = analyzer.select_topics(tokens_kept, 2, 10, passes=10, seed=42)
pd.Series(scores).plot(marker="o", figsize=(6, 3), title="c_v coherence by number of topics")
plt.xlabel("k"); plt.ylabel("c_v"); plt.show()
print("Selected k =", k_best, "| c_v =", scores[k_best])

Selected k = 8 | c_v = 0.4478


In [9]:
pd.DataFrame([(i + 1, " ".join(w)) for i, w in analyzer.topic_table(model, 10)], columns=["topic", "top words"])

,topic,top words
0,1,australian metre race indian force yacht win h...
1,2,palestinian arafat israeli attack israel hamas...
2,3,south fire test day sydney australia area peop...
3,4,bin laden force afghanistan qaeda afghan gover...
4,5,australia day centre detainee general fire act...
5,6,australian river space court people die child ...
6,7,government minister president pakistan people ...
7,8,qantas union worker company australian industr...


## 7. Limitations

- English-only models; TextBlob sentiment is a crude baseline.
- LDA depends on preprocessing and seed (fixed to 42 here); no labelled data to evaluate topics.
- Topic names are a human reading of the top words, not an output of the model.
- The corpus is short news text from a single outlet and period, so the conclusions do not generalise.